In [7]:
import os
from google import genai
from dotenv import load_dotenv

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

for m in client.models.list():
    if "generateContent" in m.supported_actions:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3.1-flash-lite
models/gemini-3-pro-image
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/gemini-4-carbon
models/gemini-4-hydrogen
models/gemini-4-neon
models/antigravity-preview-05-2026
models/antigravity-preview-09-2026
models/antigravity-preview-latest
models/deep-research-max-preview-04-2026
models/deep-research-preview-04-2026


In [12]:
%pip install google-genai pydantic python-dotenv pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import os
import json
import time
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv(override=True)
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

MODEL_ID = "models/gemini-3.1-flash-lite"

tickets = [
    {"id": 1, "text": "My payment failed twice but money got deducted!", "expected": "Billing - High"},
    {"id": 2, "text": "How do I change my profile picture?", "expected": "Account - Low"},
    {"id": 3, "text": "App crashes every time I click checkout.", "expected": "Technical - High"},
    {"id": 4, "text": "I need an invoice copy for last month's subscription.", "expected": "Billing - Low"},
    {"id": 5, "text": "Forgot my password and reset link is not arriving.", "expected": "Account - Medium"},
    {"id": 6, "text": "Website is loading very slowly on Chrome.", "expected": "Technical - Medium"},
    {"id": 7, "text": "Cancel my subscription immediately and refund my money.", "expected": "Billing - High"},
    {"id": 8, "text": "Can I add two team members to my workspace?", "expected": "Account - Low"},
    {"id": 9, "text": "API returning 500 Internal Server Error in production.", "expected": "Technical - High"},
    {"id": 10, "text": "Where can I update my GST billing address?", "expected": "Billing - Medium"}
]

ticket_list_str = "\n".join([f"{t['id']}. {t['text']}" for t in tickets])

for version in ["v1", "v2", "v3"]:
    with open(f"prompts/{version}.txt", "r", encoding="utf-8") as f:
        sys_prompt = f.read()
    
    prompt = f"{sys_prompt}\n\nClassify all 10 tickets below. Return one line per ticket in the format 'ID: Category - Priority'.\n\n{ticket_list_str}"
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.0)
    )
    
    lines = [line.strip() for line in response.text.strip().splitlines() if line.strip()]
    correct = 0
    print(f"=== Testing {version}.txt ===")
    for i, t in enumerate(tickets):
        pred_line = lines[i] if i < len(lines) else ""
        pred = pred_line.split(":", 1)[-1].strip() if ":" in pred_line else pred_line
        if pred == t["expected"]:
            correct += 1
        print(f"Expected: {t['expected']} | Predicted: {pred}")
    
    print(f"Accuracy for {version}: {correct}/10\n")
    time.sleep(2)

=== Testing v1.txt ===
Expected: Billing - High | Predicted: Billing - High
Expected: Account - Low | Predicted: Account - Low
Expected: Technical - High | Predicted: Technical - High
Expected: Billing - Low | Predicted: Billing - Low
Expected: Account - Medium | Predicted: Account - Medium
Expected: Technical - Medium | Predicted: Technical - Medium
Expected: Billing - High | Predicted: Billing - High
Expected: Account - Low | Predicted: Account - Low
Expected: Technical - High | Predicted: Technical - High
Expected: Billing - Medium | Predicted: Billing - Medium
Accuracy for v1: 10/10

=== Testing v2.txt ===
Expected: Billing - High | Predicted: Billing - High
Expected: Account - Low | Predicted: Account - Low
Expected: Technical - High | Predicted: Technical - High
Expected: Billing - Low | Predicted: Billing - Low
Expected: Account - Medium | Predicted: Account - Medium
Expected: Technical - Medium | Predicted: Technical - Medium
Expected: Billing - High | Predicted: Billing - High

In [14]:
import pandas as pd

with open("prompts/v1.txt", "r", encoding="utf-8") as f:
    base_sys_prompt = f.read()

# Edge-case (ambiguous) examples
all_examples = [
    ("I was charged twice after the app crashed during payment.", "Billing - High"),
    ("Need to transfer account ownership to a new admin.", "Account - Medium"),
    ("Export to CSV button is greyed out on Safari.", "Technical - Low"),
    ("My annual plan auto-renewed without a reminder email.", "Billing - Medium"),
    ("Two-factor authentication code is delayed by 10 minutes.", "Account - High")
]

results = []

for n in [0, 2, 3, 5]:
    few_shot_block = ""
    if n > 0:
        few_shot_block = "\n\nExamples:\n" + "\n".join(
            [f"Ticket: {ex[0]}\nOutput: {ex[1]}" for ex in all_examples[:n]]
        )
    
    prompt = f"{base_sys_prompt}{few_shot_block}\n\nClassify all 10 tickets below. Return one line per ticket in the format 'ID: Category - Priority'.\n\n{ticket_list_str}"
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.0)
    )
    
    lines = [line.strip() for line in response.text.strip().splitlines() if line.strip()]
    correct = 0
    for i, t in enumerate(tickets):
        pred_line = lines[i] if i < len(lines) else ""
        pred = pred_line.split(":", 1)[-1].strip() if ":" in pred_line else pred_line
        if pred == t["expected"]:
            correct += 1
            
    avg_tokens = round(response.usage_metadata.total_token_count / len(tickets))
    results.append({
        "Examples": f"{n}-shot",
        "Accuracy": f"{correct * 10}%",
        "Tokens per call": avg_tokens
    })
    time.sleep(2)

df_few_shot = pd.DataFrame(results)
print("=== Submission Table ===")
print(df_few_shot.to_string(index=False))

# Test effect of one wrong example on an ambiguous ticket
wrong_example_prompt = f"""{base_sys_prompt}

Examples:
Ticket: My payment failed and app crashed.
Output: Account - Low

Ticket: My payment failed twice but money got deducted!
Output:"""

res_wrong = client.models.generate_content(
    model=MODEL_ID,
    contents=wrong_example_prompt,
    config=types.GenerateContentConfig(temperature=0.0)
)
print("\n=== Effect of Wrong Example ===")
print("Expected: Billing - High | With Wrong Example:", res_wrong.text.strip())

print("""
=== When Few-Shot is Better than Zero-Shot (4 Lines) ===
1. Few-shot excels when classification boundaries are ambiguous or domain-specific edge cases exist.
2. It enforces strict, custom output formatting without needing lengthy rule descriptions.
3. Zero-shot struggles when labels depend on internal company policies rather than general knowledge.
4. Providing 2 to 3 targeted examples significantly boosts consistency at a minimal token cost.
""")

=== Submission Table ===
Examples Accuracy  Tokens per call
  0-shot     100%               28
  2-shot      80%               32
  3-shot      90%               34
  5-shot      90%               39

=== Effect of Wrong Example ===
Expected: Billing - High | With Wrong Example: Billing - High

=== When Few-Shot is Better than Zero-Shot (4 Lines) ===
1. Few-shot excels when classification boundaries are ambiguous or domain-specific edge cases exist.
2. It enforces strict, custom output formatting without needing lengthy rule descriptions.
3. Zero-shot struggles when labels depend on internal company policies rather than general knowledge.
4. Providing 2 to 3 targeted examples significantly boosts consistency at a minimal token cost.

